# Chapter 5: Stable Maps

    **Source orientation:** McDuff-Salamon, *J-holomorphic Curves and Symplectic Topology*, Chapter 5, printed pp. 115-152; PDF pp. 130-167. Sections: 5.1-5.6.

    ## Chapter Goal

    Stable maps, Gromov convergence, compactness for stable maps, uniqueness of limits, and the Gromov topology.

    The guiding question is:

    > How do bubble trees become legitimate points of a compactified moduli space?

    ## Computational Translation Guide

    | Chapter language | Computational object | Inspection target / check |
| --- | --- | --- |
| stable domain | tree of spheres with special points | each ghost has at least three |
| node | paired marked points on components | edge in dual graph |
| ghost component | constant map component | stability count |
| Gromov convergence | component maps plus neck collapse | tree limit |
| Gromov topology | neighborhood data on maps and domains | consistent labels |


## Standalone Reading Guide

    Stable maps package the output of compactness into a space with honest points. A limiting object may have several sphere components connected at nodes, and some components may carry no nonconstant image at all. Those constant components are not discarded, because they can hold marked points and remember how other components attach. Stability is the rule that prevents such ghosts from carrying continuous automorphisms.

The domain is best read through its dual graph. Vertices are components, edges are nodes, and decorations record marked points and homology classes. Gromov convergence then means more than pointwise convergence of maps: one must choose reparametrizations on components, show necks collapse to nodes, and verify that energy and markings land on the correct vertices. The uniqueness theorem says that this decorated limiting object is well-defined up to the expected equivalences.

The experiment below constructs a small stable tree. Two nonconstant components and one ghost component meet at nodes, with marked points distributed so every constant sphere has at least three special points. The figure makes clear why the stability rule is graph-theoretic at the level of domains but analytic at the level of maps.

    ## Topics In This Notebook

    - domain components, nodes, marked points, and map labels
- stability as finite automorphism control
- Gromov convergence with reparametrizations and necks
- uniqueness of stable-map limits
- topology on compactified moduli spaces

    ## Visualization Storyboard

    - A stable-map dual graph labels components, nodes, marked points, and ghost components.
- A dependency graph tracks how compactness is upgraded to a topology.
- A ledger checks the stability inequality component by component.


In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import sympy as sp


def find_book_root(start=None):
    start = (start or Path.cwd()).resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "J-Holomorphic-Curves-and-Symplectic-Topology"]:
            if (candidate / "AGENTS.md").exists() and (candidate / "utils").exists():
                return candidate
    raise RuntimeError("JHCST book root not found")


BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifact, display_artifact, save_json, save_matplotlib

UNIT = 'chapter-05'
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / UNIT
FIG_DIR = ARTIFACT_ROOT / "figures"
CHECK_DIR = ARTIFACT_ROOT / "checks"
TABLE_DIR = ARTIFACT_ROOT / "tables"
HTML_DIR = ARTIFACT_ROOT / "html"
for folder in [FIG_DIR, CHECK_DIR, TABLE_DIR, HTML_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BOOK_ROOT


## Proof Visual: Dependency Map

The graph below is a compact proof-state diagram. Read an arrow as "this idea must be under control before the next one can be used." The point is not to replace the analysis with a graph, but to keep the logical dependencies inspectable while the chapter moves between local equations, moduli spaces, compactness, and algebraic counts.


In [ ]:
CONCEPT_NODES = ['stable domain', 'node', 'ghost component', 'Gromov convergence', 'Gromov topology']
CONCEPT_EDGES = [('stable domain', 'node'), ('node', 'ghost component'), ('ghost component', 'Gromov convergence'), ('Gromov convergence', 'Gromov topology')]

G = nx.DiGraph()
G.add_nodes_from(CONCEPT_NODES)
G.add_edges_from(CONCEPT_EDGES)
pos = nx.spring_layout(G, seed=21, k=1.35)

fig, ax = plt.subplots(figsize=(9.5, 5.8))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, width=1.8, edge_color="#435466")
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2300, node_color="#eaf2f8", edgecolors="#1f567d", linewidths=1.5)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.5, font_weight="bold")
ax.set_title('Proof dependency map: Stable Maps')
ax.set_axis_off()
graph_path = save_matplotlib(fig, UNIT, "figures", "proof-dependency-map.png")
plt.close(fig)

graph_check = {
    "nodes": len(CONCEPT_NODES),
    "edges": len(CONCEPT_EDGES),
    "is_directed_acyclic_graph": nx.is_directed_acyclic_graph(G),
    "source_span": '115-152',
    "passed": len(CONCEPT_NODES) >= 5 and nx.is_directed_acyclic_graph(G),
}
graph_json = save_json(graph_check, UNIT, "checks", "proof-dependency-map.json")
display_artifact(graph_path, width=780)
graph_check


## Executable Model

This section builds a small model for one core mechanism in Stable Maps. The model is intentionally finite and inspectable: it creates an artifact, records a JSON check, and gives a learner a parameter to perturb in the applied lab.


In [ ]:
components = {
    "C0 ghost": {"marks": 1, "nodes": 2, "degree": 0},
    "C1 degree A": {"marks": 2, "nodes": 1, "degree": 1},
    "C2 degree B": {"marks": 2, "nodes": 1, "degree": 1},
}
T = nx.Graph()
T.add_edges_from([("C0 ghost", "C1 degree A"), ("C0 ghost", "C2 degree B")])
pos = {"C0 ghost": (0, 0), "C1 degree A": (-1.6, -1), "C2 degree B": (1.6, -1)}
stability = {name: data["degree"] > 0 or data["marks"] + data["nodes"] >= 3 for name, data in components.items()}
labels = {name: f"{name}\nmarks={data['marks']}, nodes={data['nodes']}" for name, data in components.items()}

fig, ax = plt.subplots(figsize=(7.5, 4.8))
colors = ["#f3d19c" if "ghost" in node else "#b7d7e8" for node in T.nodes]
nx.draw_networkx_edges(T, pos, ax=ax, width=2, edge_color="#555")
nx.draw_networkx_nodes(T, pos, ax=ax, node_size=3300, node_color=colors, edgecolors="#333")
nx.draw_networkx_labels(T, pos, labels=labels, ax=ax, font_size=8)
ax.set_title("Stable-map dual graph with a stabilized ghost component")
ax.set_axis_off()
fig_path = save_matplotlib(fig, UNIT, "figures", "stable-map-dual-graph.png")
plt.close(fig)

check = {
    "component_stability": stability,
    "all_components_stable": all(stability.values()),
    "is_tree": nx.is_tree(T),
    "total_degree": sum(data["degree"] for data in components.values()),
    "passed": bool(all(stability.values()) and nx.is_tree(T)),
}
check_path = save_json(check, UNIT, "checks", "stable-map-stability-checks.json")
display_artifact(fig_path, width=720)
check


## Invariant Ledger

The ledger records the chapter vocabulary as computational objects plus explicit checks. It is a small source map inside the notebook: every row names what should be inspected when the figure or experiment is changed.


In [ ]:
ledger_rows = [{'item': 'stable domain', 'computational_object': 'tree of spheres with special points', 'check': 'each ghost has at least three'}, {'item': 'node', 'computational_object': 'paired marked points on components', 'check': 'edge in dual graph'}, {'item': 'ghost component', 'computational_object': 'constant map component', 'check': 'stability count'}, {'item': 'Gromov convergence', 'computational_object': 'component maps plus neck collapse', 'check': 'tree limit'}, {'item': 'Gromov topology', 'computational_object': 'neighborhood data on maps and domains', 'check': 'consistent labels'}]
table_path = TABLE_DIR / "invariant-ledger.csv"
with table_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["item", "computational_object", "check"])
    writer.writeheader()
    writer.writerows(ledger_rows)

ledger_check = {
    "row_count": len(ledger_rows),
    "items": [row["item"] for row in ledger_rows],
    "has_source_specific_checks": all(row["check"] for row in ledger_rows),
    "passed": len(ledger_rows) >= 5 and all(row["check"] for row in ledger_rows),
}
ledger_json = save_json(ledger_check, UNIT, "checks", "invariant-ledger.json")
display_artifact(table_path)
ledger_check


## Applied Lab

Remove one marked point from the ghost component. The notebook's stability check will identify the exact vertex that now has a positive-dimensional automorphism group.

The intended workflow is to change one parameter, rerun the executable model, and then inspect both the figure and JSON check. If the visual impression and the invariant check disagree, trust the check first and then ask what the visualization is hiding.


## Takeaways

    - Stable maps remember both images and the limiting domain combinatorics.
- Ghost components are meaningful when they stabilize markings and nodes.
- Gromov convergence requires coordinated control of maps, domains, and labels.

    ## Sanity Checks

    The final cell asserts that the generated figures, ledgers, and JSON checks exist, are nonempty, and report successful chapter-specific invariants.


In [ ]:
expected = [
    FIG_DIR / "proof-dependency-map.png",
    FIG_DIR / 'stable-map-dual-graph.png',
    CHECK_DIR / "proof-dependency-map.json",
    CHECK_DIR / 'stable-map-stability-checks.json',
    CHECK_DIR / "invariant-ledger.json",
    TABLE_DIR / "invariant-ledger.csv",
]
for path in expected:
    min_bytes = 80 if path.suffix == ".csv" else 512
    assert_artifact(path, min_bytes=min_bytes)

for path in [CHECK_DIR / "proof-dependency-map.json", CHECK_DIR / 'stable-map-stability-checks.json', CHECK_DIR / "invariant-ledger.json"]:
    data = json.loads(path.read_text(encoding="utf-8"))
    assert data.get("passed") is True, path

print(f"Validated {len(expected)} artifacts for {UNIT}")
